# bottle_VisualAD 单张图推理

这个 notebook 用 `weight/train_on_visa/CLIP.pth` 对单张图片做 VisualAD 推理，并输出：

- 图像级 anomaly score
- `sigmoid(score)` 形式的便捷概率值
- 热力图与叠加可视化

注意：

1. 这个 notebook 复用了官方 `test.py` 的推理路径。
2. `sigmoid(score)` 只是方便观察，不等于校准后的真实概率。
3. 你本地的 `weight/train_on_visa/CLIP.pth` 如果还是 Git LFS pointer，这个 notebook 会直接报清晰错误。

In [ ]:
from pathlib import Path

# 按需修改这几个路径。
PROJECT_ROOT = Path('/Users/majingzhe/Desktop/瓶盖缺陷检测论文整理/codes/bottle_VisualAD')
CHECKPOINT_PATH = PROJECT_ROOT / 'weight/train_on_visa/CLIP.pth'
IMAGE_PATH = Path('/Users/majingzhe/Desktop/瓶盖缺陷检测论文整理/检测样本/90g/标签/Image037.PNG')

# 运行参数。
DEVICE = 'cuda:0'   # 没有可用 GPU 时改成 'cpu'
SIGMA = 4
DECISION_THRESHOLD = 0.0
SHOW_HEATMAP = True

assert PROJECT_ROOT.exists(), f'Project root not found: {PROJECT_ROOT}'
assert IMAGE_PATH.exists(), f'Image not found: {IMAGE_PATH}'
CHECKPOINT_PATH

In [ ]:
import inspect
import sys
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from scipy.ndimage import gaussian_filter

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import VisualAD_lib
from utils.transforms import get_transform
from utils.feature_transform import create_feature_transform
from utils.anomaly_detection import generate_anomaly_map_from_tokens
from utils.scoring import reduce_anomaly_map, DEFAULT_TOPK_RATIO


def ensure_real_checkpoint(path: Path):
    prefix = path.read_bytes()[:128]
    if prefix.startswith(b'version https://git-lfs.github.com/spec/v1'):
        raise RuntimeError(
            '当前 checkpoint 仍然是 Git LFS pointer，不是真实权重。请先在 bottle_VisualAD 目录下执行 `git lfs pull`，或者把真实的 CLIP.pth 放到这个路径。'
        )


def torch_load_compat(path, map_location='cpu'):
    signature = inspect.signature(torch.load)
    if 'weights_only' in signature.parameters:
        return torch.load(path, map_location=map_location, weights_only=False)
    return torch.load(path, map_location=map_location)


def load_visualad_model(checkpoint_path: Path, device: str):
    ensure_real_checkpoint(checkpoint_path)
    checkpoint = torch_load_compat(str(checkpoint_path), map_location=device)

    backbone = checkpoint.get('backbone', 'ViT-L/14@336px')
    image_size = checkpoint.get('image_size', 518)
    features_list = checkpoint.get('features_list', [6, 12, 18, 24])

    model, _ = VisualAD_lib.load(backbone, device=device)
    model.eval()
    model.to(device)

    feature_dim = model.visual.embed_dim

    model.visual.anomaly_token.data = checkpoint['anomaly_token'].to(device)
    model.visual.normal_token.data = checkpoint['normal_token'].to(device)

    layer_transforms = nn.ModuleDict()
    if 'layer_transforms' in checkpoint:
        for layer_name, state_dict in checkpoint['layer_transforms'].items():
            hidden_dim = state_dict['mlp.0.weight'].shape[0]
            layer_transforms[layer_name] = create_feature_transform(
                transform_type='mlp',
                input_dim=feature_dim,
                hidden_dim=hidden_dim,
                output_dim=feature_dim,
                dropout=0.0,
            ).to(device)
            layer_transforms[layer_name].load_state_dict(state_dict)
            layer_transforms[layer_name].eval()

    cross_attn = None
    if 'cross_attn' in checkpoint:
        from utils.spatial_cross_attention import build_layer_adaptive_cross_attention

        config = checkpoint.get('cross_attn_config', {})
        cross_attn = build_layer_adaptive_cross_attention(
            layers=features_list,
            embed_dim=feature_dim,
            num_anchors=config.get('num_anchors', 4),
            dropout=config.get('dropout', 0.1),
            res_scale_init=config.get('res_scale_init', 0.01),
        ).to(device)
        cross_attn.load_state_dict(checkpoint['cross_attn'])
        cross_attn.eval()

    preprocess, _ = get_transform(SimpleNamespace(image_size=image_size))

    return {
        'checkpoint': checkpoint,
        'model': model,
        'preprocess': preprocess,
        'cross_attn': cross_attn,
        'layer_transforms': layer_transforms,
        'backbone': backbone,
        'image_size': image_size,
        'features_list': features_list,
    }


In [ ]:
bundle = load_visualad_model(CHECKPOINT_PATH, DEVICE)
print('Backbone:', bundle['backbone'])
print('Image size:', bundle['image_size'])
print('Feature layers:', bundle['features_list'])

In [ ]:
def infer_one_image(image_path: Path, bundle, sigma=4, decision_threshold=0.0):
    model = bundle['model']
    preprocess = bundle['preprocess']
    cross_attn = bundle['cross_attn']
    layer_transforms = bundle['layer_transforms']
    features_list = bundle['features_list']
    image_size = bundle['image_size']

    device = next(model.parameters()).device
    image_pil = Image.open(image_path).convert('RGB')
    image_tensor = preprocess(image_pil).unsqueeze(0).to(device)

    with torch.no_grad():
        vision_output = model.encode_image(image_tensor, features_list)
        anomaly_features = vision_output['anomaly_features']
        normal_features = vision_output['normal_features']
        patch_tokens = vision_output['patch_tokens']
        patch_start_idx = vision_output['patch_start_idx']

        patch_features_list = [pt[:, patch_start_idx:, :] for pt in patch_tokens]
        if cross_attn is not None:
            adapted_list = cross_attn(anomaly_features, normal_features, patch_features_list, features_list)
            anomaly_features_list = [item['anomaly'] for item in adapted_list]
            normal_features_list = [item['normal'] for item in adapted_list]
        else:
            anomaly_features_list = [anomaly_features] * len(patch_tokens)
            normal_features_list = [normal_features] * len(patch_tokens)

        anomaly_map_list = []
        for idx, patch_feature in enumerate(patch_tokens):
            anomaly_feat_norm = F.normalize(anomaly_features_list[idx], dim=1, eps=1e-8)
            normal_feat_norm = F.normalize(normal_features_list[idx], dim=1, eps=1e-8)

            transform_key = f'layer_{features_list[idx]}'
            if transform_key in layer_transforms:
                batch_size, num_patches, feat_dim = patch_feature.shape
                patch_feature = layer_transforms[transform_key](patch_feature.view(-1, feat_dim)).view(batch_size, num_patches, feat_dim)

            anomaly_map = generate_anomaly_map_from_tokens(
                anomaly_feat_norm,
                normal_feat_norm,
                patch_feature[:, patch_start_idx:, :],
                image_size,
            )
            anomaly_map_list.append(anomaly_map)

        final_anomaly_map = torch.stack(anomaly_map_list).sum(dim=0).cpu()
        filtered_map = gaussian_filter(final_anomaly_map[0].numpy(), sigma=sigma)
        filtered_map_tensor = torch.from_numpy(filtered_map).unsqueeze(0)

        score = float(reduce_anomaly_map(filtered_map_tensor, mode='topk_mean', topk_ratio=DEFAULT_TOPK_RATIO).item())
        probability = float(torch.sigmoid(torch.tensor(score)).item())
        pred_is_anomaly = score >= decision_threshold

    return {
        'image_pil': image_pil,
        'anomaly_map': filtered_map,
        'score': score,
        'probability': probability,
        'pred_is_anomaly': pred_is_anomaly,
        'decision_threshold': decision_threshold,
    }


In [ ]:
result = infer_one_image(IMAGE_PATH, bundle, sigma=SIGMA, decision_threshold=DECISION_THRESHOLD)

print(f'Image: {IMAGE_PATH}')
print(f"Anomaly score: {result['score']:.6f}")
print(f"Sigmoid(score): {result['probability']:.6f}")
print(f"Decision threshold: {result['decision_threshold']:.6f}")
print('Prediction:', 'anomaly' if result['pred_is_anomaly'] else 'normal')

In [ ]:
if SHOW_HEATMAP:
    image_np = np.array(result['image_pil'])
    heatmap = result['anomaly_map']
    heatmap_norm = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)

    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    plt.imshow(image_np)
    plt.title('Input image')
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(heatmap_norm, cmap='jet')
    plt.title('Anomaly heatmap')
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(image_np)
    plt.imshow(heatmap_norm, cmap='jet', alpha=0.45)
    plt.title('Overlay')
    plt.axis('off')

    plt.tight_layout()
    plt.show()